# E8: Apply DBSCAN Clustering Algorithm to Find Data Partitions

**Dataset:** Mall_Customers.csv  
**Algorithm:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

This Google Colab notebook:
- Loads the Mall Customers dataset
- Performs basic data inspection and preprocessing
- Selects relevant numerical features
- Standardizes the features
- Applies DBSCAN clustering
- Identifies cluster labels and noise points
- Visualizes the data partitions
- Evaluates the clustering using silhouette score (when applicable)


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score


## 2. Load the Dataset

In [ ]:
# Option 1: Upload Mall_Customers.csv manually in Google Colab
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 3. Explore the Dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nStatistical summary:")
display(df.describe())


## 4. Select Features

For DBSCAN, we use:

- **Annual Income (k$)**
- **Spending Score (1-100)**

These two features are commonly used to partition Mall Customers into groups.


In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()

print("Selected features:")
display(X.head())


## 5. Standardize the Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized data:")
print(X_scaled[:5])


## 6. Apply DBSCAN Clustering

DBSCAN uses two important parameters:

- `eps`: Maximum distance between two samples for them to be considered neighbors.
- `min_samples`: Minimum number of points required to form a dense region.

The values below are a practical starting point for this dataset. You can experiment with them to observe how the partitions change.


In [ ]:
eps_value = 0.40
min_samples_value = 5

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples_value)
clusters = dbscan.fit_predict(X_scaled)

df['Cluster'] = clusters

print("DBSCAN clustering completed.")
print("\nCluster labels:")
print(np.unique(clusters))


## 7. Display Cluster Counts

In [ ]:
cluster_counts = df['Cluster'].value_counts().sort_index()

print("Number of data points in each cluster:")
print(cluster_counts)

print("\nInterpretation:")
print("-1 represents noise/outlier points identified by DBSCAN.")


## 8. Visualize the DBSCAN Data Partitions

In [ ]:
plt.figure(figsize=(10, 7))

unique_clusters = sorted(df['Cluster'].unique())

for cluster in unique_clusters:
    cluster_data = df[df['Cluster'] == cluster]

    if cluster == -1:
        label = 'Noise'
        marker = 'x'
    else:
        label = f'Cluster {cluster}'
        marker = 'o'

    plt.scatter(
        cluster_data['Annual Income (k$)'],
        cluster_data['Spending Score (1-100)'],
        label=label,
        marker=marker,
        s=60
    )

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('DBSCAN Clustering of Mall Customers')
plt.legend()
plt.grid(True)
plt.show()


## 9. Show Customers with Their Cluster Labels

In [ ]:
result = df[['CustomerID', 'Gender', 'Age',
              'Annual Income (k$)',
              'Spending Score (1-100)',
              'Cluster']]

display(result.head(20))


## 10. Identify Noise / Outlier Points

In [ ]:
noise_points = df[df['Cluster'] == -1]

print("Number of noise points:", len(noise_points))

display(
    noise_points[
        ['CustomerID', 'Gender', 'Age',
         'Annual Income (k$)',
         'Spending Score (1-100)',
         'Cluster']
    ]
)


## 11. Evaluate Clustering Using Silhouette Score

The silhouette score measures how well-separated the formed clusters are.

Noise points (`-1`) are excluded from this calculation. The score is calculated only when at least two non-noise clusters are present.


In [ ]:
non_noise_mask = clusters != -1
non_noise_labels = clusters[non_noise_mask]
non_noise_data = X_scaled[non_noise_mask]

if len(np.unique(non_noise_labels)) >= 2:
    score = silhouette_score(non_noise_data, non_noise_labels)
    print("Silhouette Score:", round(score, 4))
else:
    print("Silhouette score cannot be calculated because fewer than two non-noise clusters were formed.")


## 12. Try Different DBSCAN Parameters

In [ ]:
# Experiment with different eps values
eps_values = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]

for eps in eps_values:
    model = DBSCAN(eps=eps, min_samples=5)
    labels = model.fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)

    print(
        f"eps={eps:.2f} -> "
        f"Clusters={n_clusters}, "
        f"Noise Points={n_noise}"
    )


## 13. Final Conclusion

### Conclusion

DBSCAN was successfully applied to the **Mall Customers** dataset.

The algorithm groups customers based on the density of points in the selected feature space (**Annual Income** and **Spending Score**). Unlike algorithms such as K-Means, DBSCAN does not require the number of clusters to be specified beforehand and can identify noise/outlier points.

**Key observations:**
- Customers are divided into density-based partitions.
- Points labeled `-1` are treated as noise/outliers.
- Changing `eps` and `min_samples` changes the number and structure of clusters.
- Standardization is important because the selected features have different numerical scales.


## Viva Questions

**1. What is DBSCAN?**  
DBSCAN is a density-based clustering algorithm that groups closely packed points and identifies sparse points as noise.

**2. What does `eps` mean?**  
It is the maximum radius used to search for neighboring points.

**3. What does `min_samples` mean?**  
It is the minimum number of points required within the `eps` neighborhood to form a dense region.

**4. What does cluster label `-1` represent?**  
It represents noise or outlier points.

**5. Does DBSCAN require the number of clusters beforehand?**  
No.

**6. Why do we standardize the features?**  
To put features on comparable scales so that distance calculations are not dominated by a feature with larger numerical values.

**7. What is the main advantage of DBSCAN over K-Means?**  
DBSCAN can discover clusters without specifying the number of clusters and can identify noise points.
